# 06 — Training-time Mitigation: Mixed-Noise Augmentation

Phase 3, **axis 2** of the mitigation design: fine-tune each of the 4 NER models on
mixed-noise CoNLL-2003 and evaluate on `{clean, OCR, ASR, social}` test sets.

**Augmentation recipe** (see `scripts/train_noisy_model.py`):
- For each training sample, draw `u ~ Uniform(0, 1)`.
  - `u < 0.5` → keep clean
  - else → apply one of `{OCR, ASR, social}` noise uniformly.
- Per-sample seed `= GLOBAL_SEED + idx` for reproducibility.
- Validation + test remain clean (per-noise test sets are kept as-is from `data/noisy/`).
- 1 epoch, same hyperparameters as `01_baseline.ipynb`.

**Why subprocess-per-model?** MPS caches tensors across repeated `trainer.train()`/`trainer.evaluate()`
calls; running each model in its own Python process guarantees fresh memory. Each subprocess
writes one JSON to `results/tables/noisy_training_partials/<model>.json`. This notebook
aggregates them into a final CSV for the poster.

```
for m in bert-base-uncased bert-base-cased gpt2 google/byt5-small; do
    python scripts/train_noisy_model.py "$m"
done
```

In [1]:
import json
import os

import pandas as pd

ROOT = os.path.dirname(os.getcwd())
TABLES_DIR = os.path.join(ROOT, "results", "tables")
PARTIALS_DIR = os.path.join(TABLES_DIR, "noisy_training_partials")

MODEL_NAMES = [
    "bert-base-uncased",
    "bert-base-cased",
    "gpt2",
    "google/byt5-small",
]

## 1. Aggregate per-model partials

In [2]:
records = []
for name in MODEL_NAMES:
    safe = name.replace("/", "__")
    path = os.path.join(PARTIALS_DIR, f"{safe}.json")
    with open(path, "r", encoding="utf-8") as f:
        records.extend(json.load(f))

noisy_train_df = pd.DataFrame(records)
noisy_train_df = noisy_train_df[["model", "noise", "f1", "precision", "recall"]]
noisy_train_df.rename(columns={"f1": "f1_noisy_ft"}, inplace=True)
noisy_train_df

,model,noise,f1_noisy_ft,precision,recall
0,bert-base-uncased,clean,0.8779,0.8645,0.8916
1,bert-base-uncased,ocr,0.7647,0.7758,0.7539
2,bert-base-uncased,asr,0.8518,0.8417,0.8623
3,bert-base-uncased,social,0.7643,0.7551,0.7737
4,bert-base-cased,clean,0.8870,0.8756,0.8987
5,bert-base-cased,ocr,0.7958,0.7870,0.8047
6,bert-base-cased,asr,0.6754,0.6787,0.6721
7,bert-base-cased,social,0.7845,0.7664,0.8035
8,gpt2,clean,0.6194,0.6386,0.6013
9,gpt2,ocr,0.5230,0.5529,0.4962


## 2. Merge with clean + noisy baselines

In [3]:
# Clean baselines (from notebooks 01 + 04)
base_clean = pd.read_csv(os.path.join(TABLES_DIR, "baseline_results.csv"))[["model", "f1_clean"]]
cased_clean = pd.read_csv(os.path.join(TABLES_DIR, "bert_cased_clean_results.csv"))[["model", "f1_clean"]]
clean_df = pd.concat([base_clean, cased_clean], ignore_index=True)

# Noisy baselines (from notebooks 03 + 04)
base_noisy = pd.read_csv(os.path.join(TABLES_DIR, "noisy_evaluation.csv"))[["model", "noise", "f1"]]
cased_noisy = pd.read_csv(os.path.join(TABLES_DIR, "bert_cased_noisy_results.csv"))[["model", "noise", "f1"]]
noisy_df = pd.concat([base_noisy, cased_noisy], ignore_index=True).rename(columns={"f1": "f1_noisy"})

In [4]:
# Combine: clean row has noise='clean' with only f1_clean filled; 3 noise rows have both baselines
merged = noisy_train_df.merge(clean_df, on="model", how="left")
merged = merged.merge(noisy_df, on=["model", "noise"], how="left")

# Uplift & gap-recovery percent (for noise rows only; clean row has no uplift)
merged["f1_uplift"] = (merged["f1_noisy_ft"] - merged["f1_noisy"]).round(4)
merged["f1_recovered_pct"] = (
    (merged["f1_uplift"] / (merged["f1_clean"] - merged["f1_noisy"])) * 100
).round(2)

merged

,model,noise,f1_noisy_ft,precision,recall,f1_clean,f1_noisy,f1_uplift,f1_recovered_pct
0,bert-base-uncased,clean,0.8779,0.8645,0.8916,0.890836,NaN,NaN,NaN
1,bert-base-uncased,ocr,0.7647,0.7758,0.7539,0.890836,0.6632,0.1015,44.59
2,bert-base-uncased,asr,0.8518,0.8417,0.8623,0.890836,0.8255,0.0263,40.25
3,bert-base-uncased,social,0.7643,0.7551,0.7737,0.890836,0.6174,0.1469,53.72
4,bert-base-cased,clean,0.8870,0.8756,0.8987,0.896515,NaN,NaN,NaN
5,bert-base-cased,ocr,0.7958,0.7870,0.8047,0.896515,0.6998,0.0960,48.80
6,bert-base-cased,asr,0.6754,0.6787,0.6721,0.896515,0.2134,0.4620,67.63
7,bert-base-cased,social,0.7845,0.7664,0.8035,0.896515,0.7096,0.0749,40.07
8,gpt2,clean,0.6194,0.6386,0.6013,0.687350,NaN,NaN,NaN
9,gpt2,ocr,0.5230,0.5529,0.4962,0.687350,0.5433,-0.0203,-14.09


In [5]:
out_path = os.path.join(TABLES_DIR, "noisy_training_results.csv")
merged.to_csv(out_path, index=False)
print(f"Saved {out_path}")

Saved /Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/results/tables/noisy_training_results.csv


## 3. Combined matrix: baseline | +preprocess | +noisy-ft

Poster-ready table: one row per model × noise, five F1 columns.

In [6]:
pre_df = pd.read_csv(os.path.join(TABLES_DIR, "preprocess_results.csv"))[
    ["model", "noise", "f1_preprocess"]
]

noise_rows = merged[merged["noise"] != "clean"].copy()
combined = noise_rows.merge(pre_df, on=["model", "noise"], how="left")
combined = combined[[
    "model", "noise", "f1_clean", "f1_noisy", "f1_preprocess", "f1_noisy_ft",
]]
combined = combined.sort_values(["model", "noise"]).reset_index(drop=True)
combined.to_csv(os.path.join(TABLES_DIR, "combined_results.csv"), index=False)
print("Saved combined_results.csv")
combined

Saved combined_results.csv


,model,noise,f1_clean,f1_noisy,f1_preprocess,f1_noisy_ft
0,bert-base-cased,asr,0.896515,0.2134,0.6746,0.6754
1,bert-base-cased,ocr,0.896515,0.6998,0.7996,0.7958
2,bert-base-cased,social,0.896515,0.7096,0.7396,0.7845
3,bert-base-uncased,asr,0.890836,0.8255,0.8297,0.8518
4,bert-base-uncased,ocr,0.890836,0.6632,0.7831,0.7647
5,bert-base-uncased,social,0.890836,0.6174,0.7318,0.7643
6,google/byt5-small,asr,0.862590,0.3905,0.5901,0.6842
7,google/byt5-small,ocr,0.862590,0.7018,0.7847,0.7851
8,google/byt5-small,social,0.862590,0.7078,0.7351,0.7578
9,gpt2,asr,0.687350,0.0322,0.4620,0.3422


## 4. F1-uplift heatmap data

Pivot `f1_uplift` for the poster heatmap.

In [7]:
pivot_uplift = merged[merged["noise"] != "clean"].pivot(
    index="model", columns="noise", values="f1_uplift"
)
print("\nTraining-time F1 uplift (noisy-ft - noisy baseline):")
print(pivot_uplift.to_string())

pivot_rec = merged[merged["noise"] != "clean"].pivot(
    index="model", columns="noise", values="f1_recovered_pct"
)
print("\nGap recovery %% (uplift / (clean - noisy)):")
print(pivot_rec.to_string())


Training-time F1 uplift (noisy-ft - noisy baseline):
noise                 asr     ocr  social
model                                    
bert-base-cased    0.4620  0.0960  0.0749
bert-base-uncased  0.0263  0.1015  0.1469
google/byt5-small  0.2937  0.0833  0.0500
gpt2               0.3100 -0.0203 -0.0255

Gap recovery %% (uplift / (clean - noisy)):
noise                asr    ocr  social
model                                  
bert-base-cased    67.63  48.80   40.07
bert-base-uncased  40.25  44.59   53.72
google/byt5-small  62.21  51.81   32.30
gpt2               47.32 -14.09  -16.16


## 5. Side-by-side: preprocess vs noisy-ft

In [8]:
pre_pivot = pre_df.pivot(index="model", columns="noise", values="f1_preprocess")
ft_pivot  = merged[merged["noise"] != "clean"].pivot(index="model", columns="noise", values="f1_noisy_ft")

print("\nF1 after preprocessing (inference-time):")
print(pre_pivot.to_string())
print("\nF1 after noisy fine-tuning (training-time):")
print(ft_pivot.to_string())

diff = (ft_pivot - pre_pivot).round(4)
print("\nDifference (noisy-ft minus preprocess; + means training wins):")
print(diff.to_string())


F1 after preprocessing (inference-time):
noise                 asr     ocr  social
model                                    
bert-base-cased    0.6746  0.7996  0.7396
bert-base-uncased  0.8297  0.7831  0.7318
google/byt5-small  0.5901  0.7847  0.7351
gpt2               0.4620  0.5978  0.5436

F1 after noisy fine-tuning (training-time):
noise                 asr     ocr  social
model                                    
bert-base-cased    0.6754  0.7958  0.7845
bert-base-uncased  0.8518  0.7647  0.7643
google/byt5-small  0.6842  0.7851  0.7578
gpt2               0.3422  0.5230  0.5041

Difference (noisy-ft minus preprocess; + means training wins):
noise                 asr     ocr  social
model                                    
bert-base-cased    0.0008 -0.0038  0.0449
bert-base-uncased  0.0221 -0.0184  0.0325
google/byt5-small  0.0941  0.0004  0.0227
gpt2              -0.1198 -0.0748 -0.0395
